# ML Assignment 02
### Dataset: House Price Prediction Dataset
### Total Marks: 100

---

**Question Dataset Link:** https://www.kaggle.com/datasets/prokshitha/home-value-insights

## Student Information

In [90]:

name = "Mohammad Tuhin"
email = "md.tuhin42525@gmail.com"

print(f"Name  : {name}")
print(f"Email : {email}")

Name  : Mohammad Tuhin
Email : md.tuhin42525@gmail.com


In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [1]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import  SGDRegressor, LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

---
## Question 1 (10 Marks)

Load the House Price dataset and display:
- Dataset shape
- First 10 rows
- 5 random samples

In [51]:
df = pd.read_csv("house_price_regression_dataset.csv")


print("Dataset Shape: ", df.shape)
print("First 10 rows: ", df.head(10))
print("5 Random samples: ", df.sample(5))

Dataset Shape:  (1000, 8)
First 10 rows:     Square_Footage  Num_Bedrooms  Num_Bathrooms  Year_Built  Lot_Size  \
0            1360             2              1        1981  0.599637   
1            4272             3              3        2016  4.753014   
2            3592             1              2        2016  3.634823   
3             966             1              2        1977  2.730667   
4            4926             2              1        1993  4.699073   
5            3944             5              3        1990  2.475930   
6            3671             1              2        2012  4.911960   
7            3419             1              1        1972  2.805281   
8             630             3              3        1997  1.014286   
9            2185             4              2        1981  3.941604   

   Garage_Size  Neighborhood_Quality   House_Price  
0            0                     5  2.623829e+05  
1            1                     6  9.852609e+05  
2     

---
## Question 2 (10 Marks)

Handle missing values and perform feature engineering:
- Impute missing numerical values using `SimpleImputer` with mean strategy
- Impute missing categorical values using most frequent strategy
- Drop columns with more than 50% missing values
- Perform train-test split with `test_size=0.2` and `random_state=42`

Display the shape of final train and test sets.

In [52]:
categorical_features = df.select_dtypes(
    include= ['object']
).columns

categorical_features

Index([], dtype='object')

In [53]:
if df.isnull().sum().sum() == 0:
  print("No missing values found in dataset")

No missing values found in dataset


In [54]:
missing_percentage = df.isnull().mean() * 100
col_to_drop = missing_percentage[
    missing_percentage > 50
].index

df = df.drop(columns=col_to_drop)

In [78]:
X = df.drop(['House_Price'], axis=1)
y = df['House_Price']

In [79]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= 42)

In [59]:
numerical_features = X_train.select_dtypes(
    include=['int64', 'float64']
).columns

numerical_features

Index(['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms', 'Year_Built',
       'Lot_Size', 'Garage_Size', 'Neighborhood_Quality'],
      dtype='object')

In [60]:
num_imputer = SimpleImputer(strategy='mean')
X_train[numerical_features] = num_imputer.fit_transform(
    X_train[numerical_features]
)

X_test[numerical_features] = num_imputer.transform(
    X_test[numerical_features]
)

In [61]:
X_train

,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality
29,2028.0,2.0,3.0,1967.0,1.784790,2.0,2.0
535,3519.0,5.0,3.0,1966.0,4.009947,0.0,10.0
695,4507.0,2.0,3.0,2014.0,4.122337,0.0,7.0
557,3371.0,4.0,2.0,2000.0,1.580318,0.0,1.0
836,2871.0,5.0,1.0,1974.0,3.426914,2.0,6.0
...,...,...,...,...,...,...,...
106,2257.0,5.0,1.0,1968.0,3.131006,0.0,2.0
270,3894.0,3.0,2.0,1975.0,1.256532,0.0,5.0
860,1484.0,5.0,1.0,2010.0,1.246555,1.0,6.0
435,1865.0,4.0,2.0,1994.0,4.354220,0.0,7.0


In [62]:
print("X_train shape: ", X_train.shape)
print("X_test shape: ", X_test.shape)
print("y_train shape: ", y_train.shape)
print("y_test shape: ", y_test.shape)

X_train shape:  (800, 7)
X_test shape:  (200, 7)
y_train shape:  (800,)
y_test shape:  (200,)


---
## Question 3 (20 Marks)

Implement **Simple Linear Regression** using **only NumPy** (no Scikit-Learn allowed):
- Compute slope (`m`) and intercept (`c`) using the Batch Gradient Descent
- Predict values for the test set
- Print the learned `m` and `c` values

Use `Square_Footage` as feature (X) and `House_Price` as target (y).

In [63]:
X = df['Square_Footage'].values
y = df['House_Price'].values

In [64]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state = 42
)

In [65]:
def compute_cost(X, y, m, c):
  n = X.shape[0]
  cost = 0.0

  for i in range(n):
    prediction = m * X[i] + c
    error = prediction - y[i]
    error_squared = error ** 2
    cost = cost + error_squared

  cost = cost / (2 * n)

  return cost

In [66]:
def calcule_gradient(X, y, m, c):
  n = X.shape[0]

  dj_dw = 0.0
  dj_db = 0.0

  for i in range(n):
    prediction = m * X[i] + c
    error = prediction - y[i]

    dj_dw = dj_dw + (error * X[i])
    dj_db = dj_db + error

  return dj_dw / n, dj_db / n

In [67]:
def gradient_descent(X, y, m_input, c_input, max_iter, alpha = 1e-10):
  m = m_input
  c = c_input
  cost_memo = []
  iteration = []

  for i in range(max_iter):
    dj_dw, dj_db = calcule_gradient(X, y, m, c)

    m = m - alpha * dj_dw
    c = c - alpha * dj_db

    cost = compute_cost(X, y, m, c)
    cost_memo.append(cost)
    iteration.append(i)

    if i % 100 == 0:
      print(f"Iteration: {i}: Cost: {cost:0.4f}, m: {m:0.4f}, b: {c:0.4f}")

  return m, c, cost_memo, iteration

In [68]:
m, c, cost_memo, iterations = gradient_descent(
    X_train,
    y_train,
    0,
    0,
    1000,
    alpha = 1e-10
)

Iteration: 0: Cost: 222983803796.5385, m: 0.2055, b: 0.0001
Iteration: 100: Cost: 184574774567.4608, m: 19.8010, b: 0.0060
Iteration: 200: Cost: 152805727750.9816, m: 37.6224, b: 0.0113
Iteration: 300: Cost: 126528772673.3334, m: 53.8304, b: 0.0162
Iteration: 400: Cost: 104794460903.0682, m: 68.5709, b: 0.0207
Iteration: 500: Cost: 86817480437.4156, m: 81.9769, b: 0.0248
Iteration: 600: Cost: 71948280525.3618, m: 94.1691, b: 0.0285
Iteration: 700: Cost: 59649601865.3117, m: 105.2575, b: 0.0319
Iteration: 800: Cost: 49477064157.9598, m: 115.3420, b: 0.0349
Iteration: 900: Cost: 41063109596.8140, m: 124.5135, b: 0.0377


In [69]:
y_pred = m * X_test + c

In [70]:
print(f"Leaning Slope(m): ", m)
print(f"Learning intercept(C): ", c)

Leaning Slope(m):  132.77513449676243
Learning intercept(C):  0.040284227748360484


---
## Question 4 (10 Marks)

Build a **ColumnTransformer** that applies:
- `StandardScaler` on numerical columns: `Square_Footage`, `Num_Bedrooms`, `Num_Bathrooms`
- `OneHotEncoder` on categorical column: `Neighborhood_Quality`



In [72]:
df.head()

,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
0,1360,2,1,1981,0.599637,0,5,2.623829e+05
1,4272,3,3,2016,4.753014,1,6,9.852609e+05
2,3592,1,2,2016,3.634823,0,9,7.779774e+05
3,966,1,2,1977,2.730667,1,8,2.296989e+05
4,4926,2,1,1993,4.699073,0,8,1.041741e+06


In [73]:
#Question 4
transformer = ColumnTransformer(
    transformers= [
        ("Numerical", StandardScaler(),
        ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms']),

        ("Categorical", OneHotEncoder(), ['Neighborhood_Quality'])
    ],
    remainder='passthrough',
    verbose_feature_names_out= False
)

transformer

ColumnTransformer(remainder='passthrough',
                  transformers=[('Numerical', StandardScaler(),
                                 ['Square_Footage', 'Num_Bedrooms',
                                  'Num_Bathrooms']),
                                ('Categorical', OneHotEncoder(),
                                 ['Neighborhood_Quality'])],
                  verbose_feature_names_out=False)

## Question 5 (20 Marks)

Build a complete **Pipeline** using Scikit-Learn that includes:
- The `ColumnTransformer`
- `SGDRegressor` as the final estimator
- Train the pipeline and evaluate using RMSE and R² score
- Print predicted vs actual values for the first 10 test samples

In [75]:
print(type(X_train))
print(X_train.shape)

<class 'numpy.ndarray'>
(800,)


In [80]:
# Question 5
ETA = 0.001
MAX_ITER = 2000
ALPHA = 0.0001

sgd_regressor = Pipeline(
    steps = [
        ("ColumnTransformer", transformer),
        ("model", SGDRegressor(
            loss = "squared_error",
            penalty = 'l2',
            alpha = ALPHA,
            learning_rate='constant',
            max_iter=MAX_ITER,
            random_state= 42
        ))
    ]
)
sgd_regressor.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('ColumnTransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('Numerical', StandardScaler(),
                                                  ['Square_Footage',
                                                   'Num_Bedrooms',
                                                   'Num_Bathrooms']),
                                                 ('Categorical',
                                                  OneHotEncoder(),
                                                  ['Neighborhood_Quality'])],
                                   verbose_feature_names_out=False)),
                ('model',
                 SGDRegressor(learning_rate='constant', max_iter=2000,
                              random_state=42))])

In [81]:
def evaluate(model, name):
  train_pred = model.predict(X_train)
  test_pred = model.predict(X_test)


  print("Train r2", round(r2_score(y_train, train_pred), 4))
  print("Test r2", round(r2_score(y_test, test_pred), 4))

  print("Test RMSE", round(np.sqrt(mean_squared_error(y_train, train_pred)), 4))

  return(
      r2_score(y_test, test_pred),
      np.sqrt(mean_squared_error(y_test, test_pred))
  )

In [82]:
r2, rmse = evaluate(sgd_regressor, "SGD Regressor")

Train r2 -1.0227635353466358e+19
Test r2 -1.0165516768167197e+19
Test RMSE 810162448961409.9


In [83]:
y_pred = sgd_regressor.predict(X_test)

comparison = pd.DataFrame({
    "Actual": y_test.values[:10],
    "Predicted": y_pred[:10]
})

print(comparison)

         Actual     Predicted
0  9.010005e+05  8.221828e+14
1  4.945375e+05  8.095764e+14
2  9.494042e+05  8.003548e+14
3  1.040389e+06  8.082413e+14
4  7.940100e+05  8.156935e+14
5  7.240336e+05  8.011566e+14
6  9.984392e+05  8.142053e+14
7  9.097134e+05  8.104894e+14
8  7.926815e+05  8.125007e+14
9  9.474908e+05  8.205995e+14



---
## Question 6 (20 Marks)

Implement **Multiple Linear Regression** using **Scikit-Learn**:
- The `ColumnTransformer`
- `LinearRegression` as the final estimator
- Train the pipeline and evaluate using RMSE and R² score
- Print predicted vs actual values for the first 10 test samples

In [84]:
# Question 6
linear_regression = Pipeline(
    steps = [
        ("preprocessor", transformer),
        ("model", LinearRegression())
    ]
)


In [85]:
linear_regression.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('Numerical', StandardScaler(),
                                                  ['Square_Footage',
                                                   'Num_Bedrooms',
                                                   'Num_Bathrooms']),
                                                 ('Categorical',
                                                  OneHotEncoder(),
                                                  ['Neighborhood_Quality'])],
                                   verbose_feature_names_out=False)),
                ('model', LinearRegression())])

In [86]:
y_pred = linear_regression.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("RMSE: ", rmse)
print("R2 Score: ", r2)

RMSE:  10250.310447408485
R2 Score:  0.9983699856153558


In [87]:
y_pred = linear_regression.predict(X_test)

comparison = pd.DataFrame({
    "Actual": y_test.values[:10],
    "Predicted": y_pred[:10]
})

print(comparison)

         Actual     Predicted
0  9.010005e+05  8.698744e+05
1  4.945375e+05  4.930080e+05
2  9.494042e+05  9.435438e+05
3  1.040389e+06  1.032332e+06
4  7.940100e+05  7.761808e+05
5  7.240336e+05  7.330118e+05
6  9.984392e+05  9.923748e+05
7  9.097134e+05  8.857537e+05
8  7.926815e+05  7.962108e+05
9  9.474908e+05  9.320310e+05


---
## Question 7 (10 Marks) (You have to explore the topic and use the equation via Numpy)
### Dont use LLMs , You can use Documentation

Implement **Multiple Linear Regression** using **only NumPy**:
- Pick random 100 datas from the dataset
- Use the Normal Equation: `θ = (XᵀX)⁻¹ Xᵀy`
- Use `Square_Footage`, `Num_Bedrooms`, and `Num_Bathrooms` as features
- Print the learned coefficients (θ values)

In [88]:
# Question 7
df_sample = df.sample(n=100, random_state=42)

X = df_sample[[
    'Square_Footage',
    'Num_Bedrooms',
    'Num_Bathrooms'
]].values

y = df_sample['House_Price'].values

X = np.c_[np.ones(X.shape[0]), X]

theta = np.linalg.inv(X.T @ X) @ X.T @ y

# coefficients
print(f"Intercept       : {theta[0]}")
print(f"Square Footage  : {theta[1]}")
print(f"Num_Bedrooms    : {theta[2]}")
print(f"Num_bathrooms   : {theta[3]}")

Intercept       : 20888.259452177677
Square Footage  : 202.30820792026844
Num_Bedrooms    : 8303.281041200218
Num_bathrooms   : 3020.3285217270677
